# NovaBank Capstone Scoring, Alert Review, And Interpretability

This is the v0.9 capstone notebook for the **Digital-banking fraud detection** track at **NovaBank Digital**. It runs the full capstone loop from the scenario brief: generate the capstone dataset, build the engineered feature frame, fit a transparent scoring rule, review alerts with alert-aware metrics, choose a capacity-aware threshold, and explain *why* each alert fired — without leaving the capstone flow.

It reuses the v0.4 digital-banking feature library, the v0.1 alert-aware evaluation, the capacity-aware threshold recommender, and the v0.7 per-alert explanations. The investigation targets the `digital_scam_to_mule` and `new_beneficiary_payment` **Detection patterns** over the capstone substrate. Graph evidence (`mule_ring`) is investigative support and is covered in the synthesis notebook. Recall the glossary: the **User** is the digital login identity, distinct from the **Client** who holds the Banking relationship.

Learning goal: experience the integrated end-to-end path while keeping the limitation-aware framing visible — **a model should not be judged by headline accuracy**.

> Educational exercise on synthetic data. No real Client, account, or transaction data; no certification or legal-advice claim. The repository is pre-publication; v0.9 is a beta review point.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from banking_fraud_lab import (
    build_digital_banking_features,
    build_learner_facing_views,
    concentrate_false_positives,
    evaluate_alert_scores,
    extract_feature_importance,
    generate_capstone_digital_banking_world,
    recommend_lowest_cost_threshold,
)
from banking_fraud_lab.features import DIGITAL_BANKING_FEATURE_FAMILIES
from banking_fraud_lab.interpretability import PATTERN_TO_EXPLANATION_FAMILY
from banking_fraud_lab.schema import PROTECTED_SCENARIO_ANSWER_KEYS

pd.set_option("display.max_columns", 40)

## Generate The Capstone Dataset

The capstone uses a fixed seed and scale so every later step builds on one shared substrate. The supervised label comes from generated case outcomes for the `digital_scam_to_mule` and `new_beneficiary_payment` Detection patterns. Protected answer keys stay separate from the learner-facing views. The v0.4 families carry noisy triage outcomes, so confirmed-fraud labels are imperfect by design.

In [ ]:
tables = generate_capstone_digital_banking_world(seed=42)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

learner_summary = pd.DataFrame(
    {
        "table": ["cases", "case_outcomes", "alerts", "transactions"],
        "rows": [
            len(learner_tables["cases"]),
            len(learner_tables["case_outcomes"]),
            len(learner_tables["alerts"]),
            len(learner_tables["transactions"]),
        ],
    }
)
learner_summary

## Build The Supervised Modeling Frame

Join cases, alerts, outcomes, and the v0.4 digital-banking feature frame into one modeling frame, then fit a reproducible logistic-regression baseline on the engineered features. The feature columns come from the frozen `FeatureFamilySpec` vocabulary so the model is traceable to its Detection patterns.

In [ ]:
feature_frame = build_digital_banking_features(learner_tables)

model_frame = (
    learner_tables["cases"][["case_id", "alert_id", "transaction_id"]]
    .merge(
        learner_tables["alerts"][["alert_id", "alert_type", "severity", "reason"]],
        on="alert_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        learner_tables["case_outcomes"][[
            "case_id", "confirmed_fraud", "loss_amount_chf"
        ]],
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(feature_frame, on="transaction_id", how="inner", validate="one_to_one")
)
assert model_frame["confirmed_fraud"].nunique() == 2

feature_output_columns = [
    output_column
    for spec in DIGITAL_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
]
numeric_feature_columns = [
    column
    for column in feature_output_columns
    if pd.api.types.is_numeric_dtype(model_frame[column])
]
assert numeric_feature_columns

preprocess = ColumnTransformer(
    [("numeric", StandardScaler(), numeric_feature_columns)],
    remainder="drop",
)
baseline_model = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "model",
            LogisticRegression(
                random_state=42,
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
            ),
        ),
    ]
)
target = model_frame["confirmed_fraud"].astype(bool)
baseline_model.fit(model_frame[numeric_feature_columns], target)

model_frame = model_frame.assign(
    score=baseline_model.predict_proba(
        model_frame[numeric_feature_columns]
    )[:, 1].round(6)
)
model_frame[["alert_id", "alert_type", "confirmed_fraud", "score"]].head()

## Alert-Aware Evaluation

Alert-aware metrics report precision, recall, PR-AUC, alert volume, capacity use, and cost — never headline accuracy. The v0.4 families carry noisy triage outcomes, so confirmed-fraud labels are imperfect by design; no single metric is the ground truth. The limitation summary keeps that framing visible.

In [ ]:
alert_scores = pd.DataFrame(
    {"alert_id": model_frame["alert_id"], "score": model_frame["score"]}
)
report = evaluate_alert_scores(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    thresholds=(0.25, 0.5, 0.75),
    alert_capacity=10,
    investigation_cost_chf=75.0,
    false_positive_cost_chf=25.0,
    missed_fraud_cost_chf=1_000.0,
)
assert "accuracy is out of scope" in report["limitation_summary"]

metrics_overview = pd.DataFrame(
    [
        {"metric": "pr_auc", "value": report["pr_auc"]},
        {"metric": "limitation_summary", "value": report["limitation_summary"]},
    ]
)
metrics_overview

## Capacity-Aware Threshold Selection

Thresholds are an operational decision. The recommender sweeps alert capacity and the investigation / false-positive / missed-fraud costs so the chosen threshold reflects those tradeoffs rather than a default.

In [ ]:
threshold_recommendation = recommend_lowest_cost_threshold(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    candidate_thresholds=(0.25, 0.5, 0.75),
    alert_capacities=(5, 10, 25),
    investigation_cost_chf=75.0,
    false_positive_cost_chf=25.0,
    missed_fraud_cost_chf=1_000.0,
)

recommendation_overview = pd.DataFrame(
    [
        {
            "alert_capacity": capacity,
            "lowest_cost_threshold": entry["lowest_cost_threshold"],
            "total_cost_chf": entry["lowest_cost_summary"]["total_cost_chf"],
            "recall": entry["lowest_cost_summary"]["recall"],
        }
        for capacity, entry in threshold_recommendation["per_capacity"].items()
    ]
)
recommendation_overview

## Per-Alert Explanation: Why Was This Alert Generated?

Feature importance explains which inputs drove each alert's score. The `ExplanationFamilySpec` vocabulary ties every feature to its **Detection pattern** so the explanation is traceable to a pattern id, not a black box. Importance is an inspection aid, not proof the model is correct.

In [ ]:
scam_spec = PATTERN_TO_EXPLANATION_FAMILY["digital_scam_to_mule"]
new_beneficiary_spec = PATTERN_TO_EXPLANATION_FAMILY["new_beneficiary_payment"]

global_importance = extract_feature_importance(
    baseline_model,
    numeric_feature_columns,
    feature_frame=model_frame[numeric_feature_columns],
)

importance_overview = pd.concat(
    [
        global_importance[
            global_importance["feature"].isin(scam_spec.feature_columns)
        ].assign(detection_pattern_id="digital_scam_to_mule"),
        global_importance[
            global_importance["feature"].isin(new_beneficiary_spec.feature_columns)
        ].assign(detection_pattern_id="new_beneficiary_payment"),
    ]
)
importance_overview.sort_values("native_importance", ascending=False)

## False-Positive Concentration Review

Where do false positives fall? Grouping by `alert_type` shows whether review burden is spread evenly or concentrated on one segment — a review prompt, not a fairness verdict.

In [ ]:
fp_concentration = concentrate_false_positives(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    threshold=threshold_recommendation["recommended_threshold"],
    segment_columns=["alert_type"],
    alerts=learner_tables["alerts"],
)
fp_concentration

## Capstone Takeaways

This notebook ran the full digital-banking capstone loop on the v0.9 capstone dataset: data generation, engineered features, a fitted scoring rule, alert-aware metrics, a capacity-aware threshold, a per-alert explanation, and a false-positive review. The synthesis notebook adds graph (`mule_ring`) and case evidence as investigative support and renders the governance memo for both tracks.

**A model should not be judged by headline accuracy.** Fraud labels are sparse, alert outcomes are operational decisions, and protected answer keys stay separate from learner-facing data.